In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import sklearn

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("Seaborn version:", sns.__version__)
print("Scikit-learn version:", sklearn.__version__)

print("\nEnvironment is ready!")

Python version: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Pandas version: 3.0.5
NumPy version: 2.5.1
Matplotlib version: 3.11.1
Seaborn version: 0.13.2
Scikit-learn version: 1.9.0

Environment is ready!


In [2]:
dataset_url = (
    "https://raw.githubusercontent.com/"
    "nshomron/covidpred/master/data/"
    "corona_tested_individuals_ver_006.english.csv.zip"
)

try:
    df = pd.read_csv(
        dataset_url,
        compression="zip",
        low_memory=False
    )

    print("Dataset loaded successfully!")
    print("Dataset shape:", df.shape)

except Exception as error:
    print("Failed to load the dataset.")
    print("Error:", error)

Dataset loaded successfully!
Dataset shape: (278848, 10)


In [3]:
display(df.head())

,test_date,cough,fever,sore_throat,shortness_of_breath,head_ache,corona_result,age_60_and_above,gender,test_indication
0,2020-04-30,0.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
1,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
2,2020-04-30,0.0,1.0,0.0,0.0,0.0,negative,NaN,male,Other
3,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
4,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,male,Other


In [5]:
print("Column names:")

for column in df.columns:
    print("-", column)

Column names:
- test_date
- cough
- fever
- sore_throat
- shortness_of_breath
- head_ache
- corona_result
- age_60_and_above
- gender
- test_indication


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 278848 entries, 0 to 278847
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   test_date            278848 non-null  str    
 1   cough                278596 non-null  float64
 2   fever                278596 non-null  float64
 3   sore_throat          278847 non-null  float64
 4   shortness_of_breath  278847 non-null  float64
 5   head_ache            278847 non-null  float64
 6   corona_result        278848 non-null  str    
 7   age_60_and_above     151528 non-null  str    
 8   gender               259285 non-null  str    
 9   test_indication      278848 non-null  str    
dtypes: float64(5), str(5)
memory usage: 29.2 MB


In [7]:
missing_values = df.isnull().sum()

missing_percentage = (
    df.isnull().sum() / len(df) * 100
).round(2)

missing_table = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage (%)": missing_percentage
})

display(
    missing_table.sort_values(
        by="Missing Values",
        ascending=False
    )
)

,Missing Values,Missing Percentage (%)
age_60_and_above,127320,45.66
gender,19563,7.02
fever,252,0.09
cough,252,0.09
head_ache,1,0.00
sore_throat,1,0.00
shortness_of_breath,1,0.00
test_date,0,0.00
corona_result,0,0.00
test_indication,0,0.00


In [8]:
print("Corona result distribution:")
print(df["corona_result"].value_counts(dropna=False))

Corona result distribution:
corona_result
negative    260227
positive     14729
other         3892
Name: count, dtype: int64


In [9]:
categorical_columns = [
    "age_60_and_above",
    "gender",
    "test_indication"
]

for column in categorical_columns:
    print(f"\nColumn: {column}")
    print(df[column].value_counts(dropna=False))


Column: age_60_and_above
age_60_and_above
NaN    127320
No     125703
Yes     25825
Name: count, dtype: int64

Column: gender
gender
female    130158
male      129127
NaN        19563
Name: count, dtype: int64

Column: test_indication
test_indication
Other                     242741
Abroad                     25468
Contact with confirmed     10639
Name: count, dtype: int64


In [10]:
clean_df = df.copy()

print("Original dataset shape:", clean_df.shape)

Original dataset shape: (278848, 10)


In [11]:
text_columns = [
    "corona_result",
    "age_60_and_above",
    "gender",
    "test_indication"
]

for column in text_columns:
    clean_df[column] = (
        clean_df[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

print("Text values standardized successfully!")

Text values standardized successfully!


In [12]:
clean_df = clean_df[
    clean_df["corona_result"].isin([
        "positive",
        "negative"
    ])
].copy()

print("Records after removing 'other':", len(clean_df))

print("\nRemaining target values:")
print(clean_df["corona_result"].value_counts())

Records after removing 'other': 274956

Remaining target values:
corona_result
negative    260227
positive     14729
Name: count, dtype: int64[pyarrow]


In [13]:
clean_df["target"] = clean_df["corona_result"].map({
    "negative": 0,
    "positive": 1
})

print(clean_df[[
    "corona_result",
    "target"
]].head(10))

  corona_result  target
0      negative       0
1      negative       0
2      negative       0
3      negative       0
4      negative       0
5      negative       0
6      negative       0
7      negative       0
8      negative       0
9      negative       0


In [14]:
categorical_columns = [
    "age_60_and_above",
    "gender",
    "test_indication"
]

clean_df[categorical_columns] = (
    clean_df[categorical_columns]
    .fillna("unknown")
)

print("Categorical missing values handled!")

Categorical missing values handled!


In [15]:
symptom_columns = [
    "cough",
    "fever",
    "sore_throat",
    "shortness_of_breath",
    "head_ache"
]

for column in symptom_columns:
    # Make sure the values are numeric
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )

    # Find the most frequently occurring value
    most_common_value = clean_df[column].mode()[0]

    # Fill missing values
    clean_df[column] = (
        clean_df[column]
        .fillna(most_common_value)
        .astype(int)
    )

    print(
        column,
        "filled with:",
        most_common_value
    )

cough filled with: 0.0
fever filled with: 0.0
sore_throat filled with: 0.0
shortness_of_breath filled with: 0.0
head_ache filled with: 0.0


In [16]:
clean_df = clean_df.drop(
    columns=[
        "test_date",
        "corona_result"
    ]
)

print("Unused columns removed!")

Unused columns removed!


In [17]:
print("Cleaned dataset shape:", clean_df.shape)

print("\nMissing values after preprocessing:")
print(clean_df.isna().sum())

print("\nTarget distribution:")
print(clean_df["target"].value_counts())

display(clean_df.head())

Cleaned dataset shape: (274956, 9)

Missing values after preprocessing:
cough                  0
fever                  0
sore_throat            0
shortness_of_breath    0
head_ache              0
age_60_and_above       0
gender                 0
test_indication        0
target                 0
dtype: int64

Target distribution:
target
0    260227
1     14729
Name: count, dtype: int64


,cough,fever,sore_throat,shortness_of_breath,head_ache,age_60_and_above,gender,test_indication,target
0,0,0,0,0,0,unknown,female,other,0
1,1,0,0,0,0,unknown,female,other,0
2,0,1,0,0,0,unknown,male,other,0
3,1,0,0,0,0,unknown,female,other,0
4,1,0,0,0,0,unknown,male,other,0


In [18]:
# X contains the input features
X = clean_df.drop(columns=["target"])

# y contains the correct answer
y = clean_df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nInput features:")
print(X.columns.tolist())

X shape: (274956, 8)
y shape: (274956,)

Input features:
['cough', 'fever', 'sore_throat', 'shortness_of_breath', 'head_ache', 'age_60_and_above', 'gender', 'test_indication']


In [19]:
class_distribution = pd.DataFrame({
    "Count": y.value_counts(),
    "Percentage (%)": (
        y.value_counts(normalize=True) * 100
    ).round(2)
})

class_distribution.index = [
    "Negative (0)",
    "Positive (1)"
]

display(class_distribution)

,Count,Percentage (%)
Negative (0),260227,94.64
Positive (1),14729,5.36


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training feature shape: (219964, 8)
Testing feature shape: (54992, 8)
Training target shape: (219964,)
Testing target shape: (54992,)


In [21]:
print("Training target distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training target distribution:
target
0    94.64
1     5.36
Name: proportion, dtype: float64

Testing target distribution:
target
0    94.64
1     5.36
Name: proportion, dtype: float64


In [22]:
symptom_features = [
    "cough",
    "fever",
    "sore_throat",
    "shortness_of_breath",
    "head_ache"
]

categorical_features = [
    "age_60_and_above",
    "gender",
    "test_indication"
]

print("Symptom features:", symptom_features)
print("Categorical features:", categorical_features)

Symptom features: ['cough', 'fever', 'sore_throat', 'shortness_of_breath', 'head_ache']
Categorical features: ['age_60_and_above', 'gender', 'test_indication']


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "symptoms",
            "passthrough",
            symptom_features
        ),
        (
            "categories",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [24]:
X_train_encoded = preprocessor.fit_transform(X_train)

X_test_encoded = preprocessor.transform(X_test)

print(
    "Encoded training shape:",
    X_train_encoded.shape
)

print(
    "Encoded testing shape:",
    X_test_encoded.shape
)

Encoded training shape: (219964, 14)
Encoded testing shape: (54992, 14)


In [ ]:
encoded_feature_names = (
    preprocessor.get_feature_names_out()
)

print("Total encoded features:", len(encoded_feature_names))

for feature in encoded_feature_names:
    print("-", feature)

Total encoded features: 14
- symptoms__cough
- symptoms__fever
- symptoms__sore_throat
- symptoms__shortness_of_breath
- symptoms__head_ache
- categories__age_60_and_above_no
- categories__age_60_and_above_unknown
- categories__age_60_and_above_yes
- categories__gender_female
- categories__gender_male
- categories__gender_unknown
- categories__test_indication_abroad
- categories__test_indication_contact with confirmed
- categories__test_indication_other
